# Topic 03 pilot — Should we keep the 4-bit change?

**An engineering investigation, not a catalogue of quantizers.** Work on three independently checkable skills:

1. **Memory:** implement a tiny quantizer, explain its output error and distinguish lower-precision values from genuinely packed storage.
2. **Experiment:** compare one pinned model in FP16/NF4, then change only output length from 24 to 96. Keep raw timings, identity and separate memory counters.
3. **Decision:** make a bounded keep/revert decision using memory, time and natural answers; explain what the experiment does not establish.

Two short functions and one workload edit are yours. Infrastructure is a separate inspectable Python file, not an embedded implementation blob. **The starter intentionally fails tests until you complete those edits.** No leaderboard or grading contract changes.

Read [Book 03](https://levshaazz.github.io/llm-serving-mastery/en/book/03/). Download the pilot review bundle, or keep this notebook next to the provided `topic03_pilot.py`. In Colab, upload that file through the Files pane. No GitHub release or permission to publish is assumed by this notebook.

No GPU is needed for Part A. Part B needs a Linux CUDA GPU that you arrange yourself; availability is not guaranteed. CPU success and a teacher's recording are not your own GPU evidence.


## A1. Predict before writing code

One linear layer uses W[out,in]=[2,4] and x[in]=[4]. It produces y[out]=[2]. The eight weights and input below are invented. They are not a slice of Qwen.

For each weight: divide by scale 0.25, round to nearest with ties to even, add zero point 0, clip the integer to [−8,7], reconstruct. The signed error is reconstructed minus original. Python's `round` provides the declared tie rule.

Before running: what happens to weight 2.25? Does changing the input vector change output error even if weight error is unchanged? This affine signed-INT4 toy is **not** NF4.


In [ ]:
import math
WEIGHTS = [[-1.83, -0.52, 0.24, 2.25], [0.13, -0.88, 1.12, -2.40]]
INPUT = [1.0, -2.0, 0.5, 1.5]
PREDICTION = {
    'weight_2_25_after_reconstruction': 'TODO: your prediction',
    'will_4bit_necessarily_run_faster': 'TODO: answer and reason',
    'timing_at_24_vs_96_ids': 'TODO: predict whether the FP16/NF4 ranking or gap changes at 96, and why',
    'which_memory_counter_will_fall': 'TODO: allocated, reserved, both, or unknown; why?',
}
print(PREDICTION)


In [ ]:
def quantize_scalar(value, scale, zero_point=0):
    """Return (signed INT4 integer code, reconstructed float).

    Use round-to-nearest, ties-to-even; add zero_point; clip to [-8, 7].
    Reconstruct with scale * (code - zero_point). The toy is NOT NF4.
    Reject nonfinite values, nonpositive scales and invalid zero points.
    """
    if not math.isfinite(value) or not math.isfinite(scale) or scale <= 0:
        raise ValueError('finite value and positive finite scale required')
    if type(zero_point) is not int or not -8 <= zero_point <= 7:
        raise ValueError('signed INT4 zero point required')
    # TODO 1: three short operations; do not change the validation above.
    raise NotImplementedError('Implement round/clip/reconstruct')


In [ ]:
def test_quantizer(function):
    cases = [(1.83, .25, 0, 7, 1.75), (2.25, .25, 0, 7, 1.75),
             (-2.4, .25, 0, -8, -2.), (.625, .25, 0, 2, .5),
             (-.625, .25, 0, -2, -.5), (.6, .2, -2, 1, .6)]
    for value, scale, zero, expected_code, expected_value in cases:
        code, reconstructed = function(value, scale, zero)
        assert type(code) is int and code == expected_code, (value, code, expected_code)
        assert math.isclose(reconstructed, expected_value, abs_tol=1e-12), (value, reconstructed)
    for args in [(1., 0.), (float('nan'), .25), (1., float('inf')), (1., .25, 8)]:
        try:
            function(*args)
        except ValueError:
            continue
        raise AssertionError('Reject an invalid quantizer input: ' + repr(args))
    original = [sum(w*x for w, x in zip(row, INPUT)) for row in WEIGHTS]
    restored = [[function(w, .25)[1] for w in row] for row in WEIGHTS]
    output = [sum(w*x for w, x in zip(row, INPUT)) for row in restored]
    for observed, expected in zip([a-b for a, b in zip(output, original)], [-.705, .9]):
        assert math.isclose(observed, expected, abs_tol=1e-12)
    return {'reference': original, 'reconstructed': output,
            'signed_output_error': [a-b for a, b in zip(output, original)]}

TOY_RESULT = test_quantizer(quantize_scalar)
print(TOY_RESULT)
print('Public quantizer tests passed; explain the signs of BOTH output errors.')


## A2. Quantized values do not automatically mean packed storage

The helper below creates actual byte buffers with `struct` and `bytes`. A restored FP16 buffer still occupies 16 payload bytes. The toy packed codes take 4 payload bytes, plus 2 bytes for one FP16 scale. Shape, zero-point representation, Python-object overhead and runtime allocations are omitted.

You are **not** required to implement bit packing. Read the byte counts and explain which buffer really stores two codes per byte. This low-nibble-first two's-complement layout is not bitsandbytes NF4 packing.


In [ ]:
def storage_demo(function):
    """Actual byte buffers, excluding Python-object and metadata overhead.

    Four-byte payload uses a teaching low-nibble-first two's-complement layout;
    it is NOT the bitsandbytes NF4 layout. A separate FP16 scale takes 2 bytes.
    """
    import struct
    codes = [function(w, .25)[0] for row in WEIGHTS for w in row]
    restored = [function(w, .25)[1] for row in WEIGHTS for w in row]
    pseudo_fp16 = struct.pack('<8e', *restored)
    packed = bytes((codes[i] & 15) | ((codes[i+1] & 15) << 4) for i in range(0, 8, 2))
    return {'codes': codes, 'pseudo_fp16_bytes': len(pseudo_fp16),
            'packed_payload_bytes': len(packed), 'one_fp16_scale_bytes': 2,
            'packed_hex': packed.hex(), 'omits': 'zero point, shape, dtype and Python-object metadata'}

STORAGE = storage_demo(quantize_scalar)
assert STORAGE['pseudo_fp16_bytes'] == 16
assert STORAGE['packed_payload_bytes'] == 4
assert STORAGE['one_fp16_scale_bytes'] == 2
print(STORAGE)

# Transfer: same quantized W, different x. Does the previous output-error vector survive?
SECOND_INPUT = [0., 0., 0., 10.]
second_error = [sum((quantize_scalar(w, .25)[1]-w)*x for w, x in zip(row, SECOND_INPUT)) for row in WEIGHTS]
assert all(math.isclose(value, expected, abs_tol=1e-12) for value, expected in zip(second_error, [-5., 4.]))
print('With the second input, signed output errors:', second_error)


## A3. Calculate the timing summary yourself — still CPU only

The runner retains raw seconds. Complete your second function and pass the CPU tests before reading its precomputed summary. A factor-of-1000 mistake is an engineering error, not a cosmetic label problem. Five samples give a median and observed range here — not a reliable tail percentile.


In [ ]:
def summarize_seconds(values):
    """Return {'median_ms': ..., 'minimum_ms': ..., 'maximum_ms': ...}.

    Input contains nonempty, finite positive generate-wall times in SECONDS.
    Sort a copy. For an even count use the average of the two middle values.
    Return milliseconds, not tokens/second, p95 or HTTP TTFT.
    """
    if not values or any(type(v) not in (int, float) or not math.isfinite(v) or v <= 0 for v in values):
        raise ValueError('nonempty positive finite timings required')
    # TODO 2: calculate a median, preserve the full range, convert s -> ms.
    raise NotImplementedError('Implement the timing summary')


In [ ]:
def test_summary(function):
    samples = [.004, .001, .002, .003]
    assert function(samples) == {'median_ms': 2.5, 'minimum_ms': 1., 'maximum_ms': 4.}
    assert samples == [.004, .001, .002, .003], 'Do not mutate the raw evidence'
    assert function([.001, .003, .002])['median_ms'] == 2.
    for invalid in [[], [0.], [-1.], [float('nan')], [float('inf')], [True]]:
        try:
            function(invalid)
        except ValueError:
            continue
        raise AssertionError('Reject invalid timings: ' + repr(invalid))

test_summary(summarize_seconds)
print('Timing-summary tests passed.')


## B1. Write the experimental contract

We compare `Qwen/Qwen2.5-0.5B-Instruct` at immutable revision `7ae557604adf67be50417f59c2c2f167def9a775`. Both use batch 1, greedy generation, SDPA, KV caching, the same tokenizer/chat wrapper and FP16 computation. NF4 uses bitsandbytes, FP16 compute and double quantization off. This changes **representation and execution path**, not bit width alone.

There are two deliberately separate workloads:

- **Timing intervention:** literal prompt `Continue the sequence with comma-separated integers and no explanation: 2, 4, 6,`; force exactly 24 or 96 output IDs. Five raw trials per length and format, after one excluded warmup for each condition. Length order alternates within a format. This is a synthetic timing workload, not a quality answer or support-service latency benchmark.
- **Quality smoke:** twelve handwritten, labelled support tickets (four each: billing/access/bug); natural stopping, maximum 12 IDs. Score an answer only if stripping surrounding whitespace yields the exact lowercase label. Inspect every mismatch. This tiny synthetic set does not certify real-world accuracy. Six public transfer cases are kept out of the default run; they are not a secret evaluation set.

The GPU clock brackets `generate` with CUDA synchronization. It excludes model loading, tokenization and input transfer, and includes host dispatch. No HTTP TTFT, p95 or throughput claim. Formats load sequentially: FP16 then NF4, a time-order confound. A reverse-order repeat is optional only after diagnosing the first run.

**Your intervention:** keep baseline length 24 and change only the second value to 96. Do not change the model, prompt, batch, sampler or package pins at the same time.


In [ ]:
def check_lengths(lengths):
    assert type(lengths) in (list, tuple) and list(lengths) == [24, 96], (
        'Keep the baseline 24; change only the output cap to 96 for the second condition.')

LENGTHS = [24, 24]  # TODO 3: change ONLY the second output length to 96.
check_lengths(LENGTHS)
TRIALS = 5
assert TRIALS == 5


## B2. Inspect and verify the infrastructure file

This file contains the public synthetic cases, safety gates, model setup, clocks, profiler and artifact validator. You do not need to understand all infrastructure at once. Inspect three small entry points and answer a concrete question:

1. `score_output`: is `Billing.` a correct answer? Which transformation is allowed?
2. The nested `generate` inside `run`: where are the two CUDA synchronizations, and which work lies outside the clock?
3. `validate_artifact`: which equality prevents comparing different tokenized prompts across formats?

The SHA below binds this notebook to reviewed bytes. Use the matching bundle; do not bypass a mismatch or fetch an unpinned `main` file.

The runner is standard-library-only on import. Inference executes in a separate process, not in the Colab kernel. Runtime package pins live in an isolated venv using the host Torch/CUDA installation; the base Colab environment is not modified. Colab may lack `ensurepip`, so setup uses `--without-pip` and the inherited installer, after verifying that its install target is the dedicated venv.


In [ ]:
from pathlib import Path
import hashlib, importlib.util, json, subprocess, sys

RUNNER_CANDIDATES = [Path('topic03_pilot.py'), Path('../teaching/topic03_pilot.py'), Path('teaching/topic03_pilot.py')]
RUNNER = next((path.resolve() for path in RUNNER_CANDIDATES if path.is_file()), None)
assert RUNNER is not None, 'Upload the provided topic03_pilot.py through Colab Files, then rerun this cell.'
EXPECTED_RUNNER_SHA256 = 'd40cdaa393fe4122ced0a7f67a8dc86a42b452e4d14c535fee2e828d00422c87'
assert hashlib.sha256(RUNNER.read_bytes()).hexdigest() == EXPECTED_RUNNER_SHA256, 'Wrong runner revision: use matching bundle.'
spec = importlib.util.spec_from_file_location('topic03_pilot_infrastructure', RUNNER)
infra = importlib.util.module_from_spec(spec)
spec.loader.exec_module(infra)
assert infra.PROTOCOL == 'topic03-engineering-pilot-v1'
print('Verified protocol:', infra.PROTOCOL)
for case in infra.CASES:
    print(case['id'], repr(case['text']), '->', case['label'])
print('Public transfer cases excluded from this run:', len(infra.TRANSFER_CASES))


## B3. Prepare a bounded CUDA run — explicit opt-in

Startup requires an otherwise idle GPU, at least **5 GiB free host RAM**, **5 GiB free GPU memory**, and 3 GiB free cache-disk space if downloading is needed. During work: at least 2 GiB free host RAM and 1 GiB free GPU memory. These are conservative gates for this new 0.5B pilot, **not** relaxed gates for the historical v3 protocol. They are sampled headroom checks, not an OS memory sandbox or a guarantee on every Colab machine.

Reuse the pinned model cache. Set `LSM_MODEL_CACHE` only if you know your existing cache path; otherwise the normal Hugging Face cache is reused. `HF_HUB_OFFLINE=1` refuses missing weights. Do not delete models between attempts, mount personal Drive unnecessarily, or run other GPU notebooks concurrently.

The worker has a 600-second wall budget; the parent allows 45 additional seconds for teardown. If OOM, timeout, failure or dirty cleanup occurs, stop and inspect the retained partial result. A failed capture is not complete evidence. Verify process/GPU state before retrying. No background server is started.


In [ ]:
RUN_GPU = False  # Explicit opt-in only after the CPU tests and prediction are complete.
ENV = Path('/content/lsm-topic03-pilot-env') if Path('/content').exists() else Path('.venv-topic03-pilot')
WORKER_PYTHON = ENV / 'bin' / 'python'
if RUN_GPU:
    test_quantizer(quantize_scalar)
    test_summary(summarize_seconds)
    check_lengths(LENGTHS)
    print('Pre-install headroom:', infra.resource_sample(require_idle=True, startup=True))
    # Reconfigure this dedicated venv even if a prior ensurepip attempt left a Python binary.
    # No --clear: preserve installed matching packages. Never bootstrap or install into base Python.
    subprocess.check_call([sys.executable, '-m', 'venv', '--without-pip', '--system-site-packages', str(ENV)])
    environment_probe = "import json,sys,sysconfig,pip; print(json.dumps({'prefix':sys.prefix,'base_prefix':sys.base_prefix,'purelib':sysconfig.get_paths()['purelib']}))"
    worker_environment = json.loads(subprocess.check_output([str(WORKER_PYTHON), '-c', environment_probe], text=True))
    assert Path(worker_environment['prefix']).resolve() == ENV.resolve(), 'Worker must use the dedicated venv'
    assert worker_environment['prefix'] != worker_environment['base_prefix'], 'Never install into base Python'
    assert Path(worker_environment['purelib']).resolve().is_relative_to(ENV.resolve()), 'Install target must stay inside the venv'
    version_probe = "import importlib.metadata as m,json; names=" + repr(list(infra.PINS)) + "; result={}; "
    version_probe += "\nfor n in names:\n try: result[n]=m.version(n)\n except m.PackageNotFoundError: result[n]=None\nprint(json.dumps(result))"
    installed = json.loads(subprocess.check_output([str(WORKER_PYTHON), '-c', version_probe], text=True))
    missing = [name + '==' + wanted for name, wanted in infra.PINS.items() if installed.get(name) != wanted]
    if missing:
        subprocess.check_call([str(WORKER_PYTHON), '-m', 'pip', '--isolated', 'install', *missing])
    print('Isolated worker packages prepared; host Torch/CUDA will be recorded, not silently replaced.')
else:
    print('GPU and package installation are OFF. Complete the CPU work first.')


In [ ]:
from datetime import datetime, timezone
ATTEMPT = None
if RUN_GPU:
    check_lengths(LENGTHS)
    ATTEMPT = Path('evidence') / datetime.now(timezone.utc).strftime('topic03-pilot-%Y%m%dT%H%M%S%fZ')
    ATTEMPT.mkdir(parents=True, exist_ok=False)
    command = [str(WORKER_PYTHON), str(RUNNER), '--output', str(ATTEMPT/'result.json'),
               '--lengths', *map(str, LENGTHS), '--trials', str(TRIALS), '--timeout', '600']
    subprocess.run(command, check=True, timeout=645)
    # Separate post-process query; do not publish account/PID details from the notebook.
    print(subprocess.check_output(['nvidia-smi', '--id=0', '--query-gpu=memory.used',
          '--format=csv,noheader,nounits'], text=True).strip(), 'MiB device memory after worker exit')
    infra.resource_sample(require_idle=True)
    print('Preserved attempt:', ATTEMPT)
else:
    print('Not run: no new GPU evidence exists.')


## C1. Compare your summary with the retained raw evidence

Now apply the CPU function you already tested to every format/length group. Preserve seconds in the raw artifact and label the converted summary in milliseconds. Inspect natural quality answers separately.


In [ ]:
EVIDENCE_FILE = ATTEMPT/'result.json' if ATTEMPT is not None else None
RESULT = None
if EVIDENCE_FILE is not None:
    RESULT = json.loads(EVIDENCE_FILE.read_text())
    assert infra.validate_artifact(RESULT)
    for label, entry in RESULT['formats'].items():
        for length in LENGTHS:
            rows = [row for row in entry['timing_trials'] if row['length'] == length]
            own = summarize_seconds([row['wall_s'] for row in rows])
            official = entry['summaries'][str(length)]
            assert all(math.isclose(value, official[key], rel_tol=1e-12) for key, value in own.items())
            print(label, length, 'raw seconds:', [row['wall_s'] for row in rows])
            print('Your summary:', own)
            print('Peak allocated MiB:', official['peak_allocated_bytes']/2**20,
                  'peak reserved MiB:', official['peak_reserved_bytes']/2**20)
        print('Natural quality:', entry['quality_summary'])
        for row in entry['quality_samples']:
            print(row['case_id'], repr(row['output']), row['stop_reason'],
                  'valid:', row['valid_label'], 'correct:', row['correct'])
else:
    print('No artifact loaded. Do not replace your run with fabricated numbers.')


## C2. Read a profile without inventing a cause

The runner takes one 24-ID profile per format **after** its timing trials. Examine `profile.status` first. If unavailable, report that limitation. If complete, find an operator that changes and write one testable hypothesis. Inclusive CPU/device times can overlap; summing them does not produce wall time. A `Linear4bit` module count verifies a treatment, not an accelerated kernel.

The profiler summary is not a measurement of HBM bandwidth, achieved occupancy or pure dequantization overhead. It does not, by itself, prove why the wall time changed. Reserved allocator pools persist after warmup and between lengths: they are not independent fresh-memory experiments.


In [ ]:
if RESULT is not None:
    for label, entry in RESULT['formats'].items():
        print(label, 'quantized linear modules:', entry['quantized_linear_modules'])
        print(json.dumps(entry['profile'], indent=2))
else:
    print('No profile to interpret; do not infer kernels from the toy or module names.')


## C3. Submit a bounded decision, not a victory slide

Keep your modified notebook (or a readable diff of the two functions and length edit), original attempt `result.json`, one plot/table from raw timings, and a half-page decision memo. This pilot is a teaching artifact, not a replacement for existing judge submission files.

**Before disconnecting Colab:** download the preserved `result.json` from the Files pane (right-click → Download), then use File → Download → Download .ipynb to keep your edited notebook. Colab's temporary VM is not durable storage. Keep the runner hash and model/runtime metadata with the JSON. Check notebook outputs for tokens, private paths or account data before sharing; submit student work privately, not as a public Issue. Do not download the model cache as a deliverable. If a run failed, preserve its partial JSON separately rather than calling it a completed comparison.

Your memo must state:

1. What you predicted and one prediction that the evidence changed. Show both toy output errors and explain the storage bytes.
2. Model revision, hardware/runtime, batch, prompt, stopping mode, trial count and exactly what changed. Separate historical records from your own run.
3. Raw time range and memory-counter comparison at 24 and 96 IDs. Do not call allocated/reserved total VRAM or generalize synthetic sequence timings to support-service latency.
4. All natural-output mismatches (including invalid labels/truncation), the small synthetic denominator and what remains unknown. There is no accuracy guarantee or production threshold here.
5. Keep FP16, keep NF4 conditionally, or postpone the choice — name the constraint and evidence. An optimization that is correctly rejected is a successful investigation. State one discriminating next experiment, not five more algorithms.

**Transfer check:** before touching the six public transfer tickets in `infra.TRANSFER_CASES`, write what failure you expect; do not tune the core prompt on them. They are an exercise, not a hidden grading set. An optional extension may run them separately after the required artifact is preserved. No automatic extra GPU run is required.

### Optional, not hidden mandatory work

AWQ/GPTQ derivations, NF4 codebook internals, double quantization, FP8/NVFP4, full packing implementation, reverse-format-order replication and custom kernels are outside this pilot's mandatory core. The 15-topic course framework remains; overload is not moved into mandatory homework.


## Sources and status

- Pedagogical idea: [MIT EfficientML Lab 4](https://colab.research.google.com/drive/16H9RvSg4XIF35X3fLGQUVwAE9ccvDj14), controlled comparisons. Its pseudo-quantization example does not demonstrate packed storage or kernel acceleration. Our code and support cases are original.
- Pedagogical idea: [CMU Deep Learning Systems](https://dlsyscourse.org/), small student functions inside a cumulative project. We do not adopt its full prerequisite level.
- Runtime behavior: [Transformers 4.51.3 bitsandbytes guide](https://huggingface.co/docs/transformers/v4.51.3/en/quantization/bitsandbytes), [bitsandbytes 0.50.2 source](https://github.com/bitsandbytes-foundation/bitsandbytes/tree/0.50.2), [PyTorch CUDA memory semantics](https://docs.pytorch.org/docs/stable/notes/cuda.html#cuda-memory-management), [PyTorch profiler](https://docs.pytorch.org/docs/stable/profiler.html).
- The [2 October historical v3 capture](https://levshaazz.github.io/llm-serving-mastery/seminars/runs/2026-10-02-topic03-quantization/README.md) used a different four-prompt protocol and instructor hardware. It is not this pilot, not a comparison against your Colab device, and not your execution.

This is a pilot awaiting learner evaluation. Automated checks can verify arithmetic, identity and artifact completeness; they cannot prove that a student can independently transfer the skill.
